# Filter vs Vector: WMA and HMA

Compares `cdef class` streaming filters (one loop and no full-sized intermediate arrays) against the current `calc_*` vector kernels.

Three filter variants:
- **base class** — `process()` calls `self.update()` through virtual dispatch
- **no base class** — direct `cdef _update()` calls
- **inline** — direct `cdef inline _update()` calls

In [1]:
import timeit
import numpy as np

%load_ext cython

from mintalib import core
from mintalib.samples import sample_prices

prices = sample_prices()
close = prices.close.values.astype(float)
print(f"Rows: {len(close):,}")

Rows: 11,504


In [2]:
%%cython -c=-Wno-unreachable-code
# cython: boundscheck=False, wraparound=False, cdivision=True, nonecheck=False
# Variant 1: base class and virtual update dispatch

import math
import numpy as np
from libc.math cimport isnan

cdef double NAN1 = float('nan')

cdef class Filter:
    cpdef double update(self, double x) except *:
        return NAN1

    cpdef reset(self):
        pass

    cpdef process(self, const double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.empty(size, float)
        cdef double[:] output = result
        cdef long i
        self.reset()
        for i in range(size):
            output[i] = self.update(xs[i])
        return result

cdef class WmaFilter(Filter):
    cdef readonly long period
    cdef double wdiv, rsum, wsum
    cdef long count, index
    cdef object buffer_obj
    cdef double[:] buffer

    def __init__(self, long period):
        self.period = period
        self.wdiv = period * (period + 1) / 2.0
        self.buffer_obj = np.empty(period, float)
        self.buffer = self.buffer_obj
        self.reset()

    cpdef reset(self):
        self.rsum = self.wsum = 0.0
        self.count = self.index = 0

    cpdef double update(self, double x) except *:
        cdef double old
        if isnan(x):
            self.reset()
            return NAN1
        old = self.buffer[self.index]
        self.buffer[self.index] = x
        self.index += 1
        if self.index == self.period:
            self.index = 0
        self.count += 1
        self.rsum += x
        self.wsum += self.count * x
        if self.count > self.period:
            self.wsum -= self.rsum
            self.rsum -= old
            self.count -= 1
        if self.count == self.period:
            return self.wsum / self.wdiv
        return NAN1

cdef class HmaFilter(Filter):
    cdef readonly long period
    cdef WmaFilter half, full, final

    def __init__(self, long period):
        self.period = period
        self.half = WmaFilter(round(period / 2))
        self.full = WmaFilter(period)
        self.final = WmaFilter(round(math.sqrt(period)))

    cpdef reset(self):
        self.half.reset()
        self.full.reset()
        self.final.reset()

    cpdef double update(self, double x) except *:
        cdef double h = self.half.update(x)
        cdef double f = self.full.update(x)
        if isnan(h) or isnan(f):
            return self.final.update(NAN1)
        return self.final.update(2.0 * h - f)

print("Done (base class)!")

Done (base class)!


In [3]:
%%cython -c=-Wno-unreachable-code
# cython: boundscheck=False, wraparound=False, cdivision=True, nonecheck=False
# Variants 2 and 3: direct C calls, with and without inline

import math
import numpy as np
from libc.math cimport isnan

cdef double NAN2 = float('nan')

cdef class WmaFilter2:
    cdef readonly long period
    cdef double wdiv, rsum, wsum
    cdef long count, index
    cdef object buffer_obj
    cdef double[:] buffer

    def __init__(self, long period):
        self.period = period
        self.wdiv = period * (period + 1) / 2.0
        self.buffer_obj = np.empty(period, float)
        self.buffer = self.buffer_obj
        self._reset()

    cdef void _reset(self) noexcept:
        self.rsum = self.wsum = 0.0
        self.count = self.index = 0

    cdef double _update(self, double x) noexcept:
        cdef double old
        if isnan(x):
            self._reset()
            return NAN2
        old = self.buffer[self.index]
        self.buffer[self.index] = x
        self.index = (self.index + 1) % self.period
        self.count += 1
        self.rsum += x
        self.wsum += self.count * x
        if self.count > self.period:
            self.wsum -= self.rsum
            self.rsum -= old
            self.count -= 1
        if self.count == self.period:
            return self.wsum / self.wdiv
        return NAN2

    def process(self, const double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.empty(size, float)
        cdef double[:] output = result
        cdef long i
        self._reset()
        for i in range(size):
            output[i] = self._update(xs[i])
        return result

cdef class HmaFilter2:
    cdef WmaFilter2 half, full, final

    def __init__(self, long period):
        self.half = WmaFilter2(round(period / 2))
        self.full = WmaFilter2(period)
        self.final = WmaFilter2(round(math.sqrt(period)))

    def process(self, const double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.empty(size, float)
        cdef double[:] output = result
        cdef double h, f
        cdef long i
        self.half._reset(); self.full._reset(); self.final._reset()
        for i in range(size):
            h = self.half._update(xs[i])
            f = self.full._update(xs[i])
            if isnan(h) or isnan(f):
                output[i] = self.final._update(NAN2)
            else:
                output[i] = self.final._update(2.0 * h - f)
        return result

cdef class WmaFilter3:
    cdef readonly long period
    cdef double wdiv, rsum, wsum
    cdef long count, index
    cdef object buffer_obj
    cdef double[:] buffer

    def __init__(self, long period):
        self.period = period
        self.wdiv = period * (period + 1) / 2.0
        self.buffer_obj = np.empty(period, float)
        self.buffer = self.buffer_obj
        self._reset()

    cdef inline void _reset(self) noexcept:
        self.rsum = self.wsum = 0.0
        self.count = self.index = 0

    cdef inline double _update(self, double x) noexcept:
        cdef double old
        if isnan(x):
            self._reset()
            return NAN2
        old = self.buffer[self.index]
        self.buffer[self.index] = x
        self.index = (self.index + 1) % self.period
        self.count += 1
        self.rsum += x
        self.wsum += self.count * x
        if self.count > self.period:
            self.wsum -= self.rsum
            self.rsum -= old
            self.count -= 1
        if self.count == self.period:
            return self.wsum / self.wdiv
        return NAN2

    def process(self, const double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.empty(size, float)
        cdef double[:] output = result
        cdef long i
        self._reset()
        for i in range(size):
            output[i] = self._update(xs[i])
        return result

cdef class HmaFilter3:
    cdef WmaFilter3 half, full, final

    def __init__(self, long period):
        self.half = WmaFilter3(round(period / 2))
        self.full = WmaFilter3(period)
        self.final = WmaFilter3(round(math.sqrt(period)))

    def process(self, const double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.empty(size, float)
        cdef double[:] output = result
        cdef double h, f
        cdef long i
        self.half._reset(); self.full._reset(); self.final._reset()
        for i in range(size):
            h = self.half._update(xs[i])
            f = self.full._update(xs[i])
            if isnan(h) or isnan(f):
                output[i] = self.final._update(NAN2)
            else:
                output[i] = self.final._update(2.0 * h - f)
        return result

print("Done (direct and inline)!")

Done (direct and inline)!


In [4]:
period = 20

pairs = [
    ("WMA  (base)", WmaFilter(period).process(close), core.calc_wma(close, period)),
    ("WMA  (direct)", WmaFilter2(period).process(close), core.calc_wma(close, period)),
    ("WMA  (inline)", WmaFilter3(period).process(close), core.calc_wma(close, period)),
    ("HMA  (base)", HmaFilter(period).process(close), core.calc_hma(close, period)),
    ("HMA  (direct)", HmaFilter2(period).process(close), core.calc_hma(close, period)),
    ("HMA  (inline)", HmaFilter3(period).process(close), core.calc_hma(close, period)),
]

for label, actual, expected in pairs:
    np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12, equal_nan=True)
    print(f"{label}: OK")

WMA  (base): OK
WMA  (direct): OK
WMA  (inline): OK
HMA  (base): OK
HMA  (direct): OK
HMA  (inline): OK


In [5]:
def bench(fn, repeat=7, number=100):
    return min(timeit.repeat(fn, repeat=repeat, number=number)) / number

def fmt(seconds):
    return f"{seconds * 1_000:.3f} ms"

wma1 = WmaFilter(period); wma2 = WmaFilter2(period); wma3 = WmaFilter3(period)
hma1 = HmaFilter(period); hma2 = HmaFilter2(period); hma3 = HmaFilter3(period)
cases = [
    ("WMA", wma1.process, wma2.process, wma3.process, lambda: core.calc_wma(close, period)),
    ("HMA", hma1.process, hma2.process, hma3.process, lambda: core.calc_hma(close, period)),
]

print(f"Rows: {len(close):,}  period={period}\n")
print(f"{'':6}  {'base':>10}  {'direct':>10}  {'inline':>10}  {'vector':>10}")
print("-" * 58)
for name, f_base, f_direct, f_inline, f_vector in cases:
    times = [bench(lambda f=f: f(close)) for f in (f_base, f_direct, f_inline)]
    times.append(bench(f_vector))
    print(f"{name:<6}  " + "  ".join(f"{fmt(t):>10}" for t in times))

Rows: 11,504  period=20

              base      direct      inline      vector
----------------------------------------------------------


WMA       0.064 ms    0.064 ms    0.064 ms    0.026 ms


HMA       0.141 ms    0.085 ms    0.066 ms    0.085 ms


## Stack-allocated C state

Use a C struct for rolling WMA state and a raw pointer to a caller-owned circular buffer. HMA composes three independent WMA states in one pass.

In [6]:
%%cython -c=-Wno-unreachable-code
# cython: boundscheck=False, wraparound=False, cdivision=True, nonecheck=False

import math
import numpy as np
from libc.math cimport isnan

cdef double NAN3 = float('nan')

cdef struct WmaState:
    long period
    long count
    long index
    double wdiv
    double rsum
    double wsum
    double* buffer

cdef inline void wma_init(WmaState* state, long period, double* buffer) noexcept nogil:
    state.period = period
    state.count = 0
    state.index = 0
    state.wdiv = period * (period + 1) / 2.0
    state.rsum = 0.0
    state.wsum = 0.0
    state.buffer = buffer

cdef inline double wma_update(WmaState* state, double x) noexcept nogil:
    cdef double old
    if isnan(x):
        state.count = 0
        state.index = 0
        state.rsum = 0.0
        state.wsum = 0.0
        return NAN3
    old = state.buffer[state.index]
    state.buffer[state.index] = x
    state.index += 1
    if state.index == state.period:
        state.index = 0
    state.count += 1
    state.rsum += x
    state.wsum += state.count * x
    if state.count > state.period:
        state.wsum -= state.rsum
        state.rsum -= old
        state.count -= 1
    if state.count == state.period:
        return state.wsum / state.wdiv
    return NAN3

def wma_struct(const double[:] xs, long period):
    cdef long size = xs.shape[0]
    cdef object result = np.empty(size, float)
    cdef double[:] output = result
    cdef object buffer_array = np.empty(period, float)
    cdef double[::1] buffer = buffer_array
    cdef WmaState state
    cdef long i
    wma_init(&state, period, &buffer[0])
    with nogil:
        for i in range(size):
            output[i] = wma_update(&state, xs[i])
    return result

def hma_struct(const double[:] xs, long period):
    if period == 1:
        return np.asarray(xs).copy()
    cdef long half_period = round(period / 2)
    cdef long final_period = round(math.sqrt(period))
    cdef long size = xs.shape[0]
    cdef object result = np.empty(size, float)
    cdef double[:] output = result
    cdef object half_array = np.empty(half_period, float)
    cdef object full_array = np.empty(period, float)
    cdef object final_array = np.empty(final_period, float)
    cdef double[::1] half_buffer = half_array
    cdef double[::1] full_buffer = full_array
    cdef double[::1] final_buffer = final_array
    cdef WmaState half, full, final
    cdef double h, f
    cdef long i
    wma_init(&half, half_period, &half_buffer[0])
    wma_init(&full, period, &full_buffer[0])
    wma_init(&final, final_period, &final_buffer[0])
    with nogil:
        for i in range(size):
            h = wma_update(&half, xs[i])
            f = wma_update(&full, xs[i])
            if isnan(h) or isnan(f):
                output[i] = wma_update(&final, NAN3)
            else:
                output[i] = wma_update(&final, 2.0 * h - f)
    return result

In [7]:
struct_cases = [
    ("WMA", wma_struct, WmaFilter3(period).process, core.calc_wma),
    ("HMA", hma_struct, HmaFilter3(period).process, core.calc_hma),
]

print(f"Rows: {len(close):,}  period={period}\n")
print(f"{'':6}  {'C struct':>10}  {'object':>10}  {'vector':>10}")
print("-" * 46)
for name, struct_fn, object_fn, vector_fn in struct_cases:
    actual = struct_fn(close, period)
    expected = vector_fn(close, period)
    np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12, equal_nan=True)
    t_struct = bench(lambda f=struct_fn: f(close, period))
    t_object = bench(lambda f=object_fn: f(close))
    t_vector = bench(lambda f=vector_fn: f(close, period))
    print(f"{name:<6}  {fmt(t_struct):>10}  {fmt(t_object):>10}  {fmt(t_vector):>10}")

Rows: 11,504  period=20

          C struct      object      vector
----------------------------------------------
WMA       0.024 ms    0.065 ms    0.026 ms


HMA       0.034 ms    0.065 ms    0.084 ms
